In [1]:
# %pip install detoxify --quiet

from detoxify import Detoxify
from typing import Dict, Tuple


# Toxicity Detection using Detoxify

This notebook demonstrates the optimized toxicity detection model using Detoxify.

## Performance Notes

**First Run**: The model (~500MB) will download from HuggingFace and be cached in `~/.cache/huggingface/`. This can take 30-60 seconds depending on your internet connection.

**Subsequent Runs**: The model loads from cache, taking only 2-5 seconds.

**Optimization**: The model uses a singleton pattern, so it's only loaded once per session regardless of how many times you create detector instances.


In [ ]:
# Test model loading performance
import time

print("Testing ToxicityDetector initialization...")
start = time.time()

# First instance - will initialize the model
detector1 = ToxicityDetector()
init_time = time.time() - start
print(f"✓ First initialization took: {init_time:.2f} seconds")

# Second instance - should reuse existing model
start = time.time()
detector2 = ToxicityDetector()
reuse_time = time.time() - start
print(f"✓ Second initialization took: {reuse_time:.4f} seconds (reused existing model)")

# Test inference time
test_text = "This is a test message."
start = time.time()
scores, label = detector1.score(test_text)
inference_time = time.time() - start
print(f"✓ Inference took: {inference_time:.4f} seconds")

print(f"\nBoth detector instances are the same object: {detector1 is detector2}")


In [ ]:
import torch

class ToxicityDetector:
    """
    Detoxify-based toxicity detector using unitary/toxic-bert model.
    
    Detects multiple toxicity categories:
        - toxicity: overall toxicity score
        - severe_toxicity: severe toxic content
        - obscene: obscene language
        - threat: threatening content
        - insult: insulting content
        - identity_attack: attacks on identity groups
    
    More info: https://github.com/unitaryai/detoxify
    """

    _instance = None  # Singleton cache
    _initialized = False  # Track initialization status

    LABELS = [
        "toxicity",
        "severe_toxicity",
        "obscene",
        "threat",
        "insult",
        "identity_attack"
    ]

    def __new__(cls):
        if cls._instance is None:
            print("[ToxicityDetector] Creating new instance...")
            cls._instance = super().__new__(cls)
        return cls._instance
    
    def __init__(self):
        # Only initialize once, even if __init__ is called multiple times
        if not ToxicityDetector._initialized:
            print("[ToxicityDetector] Initializing model (this may take a moment on first run)...")
            
            # Set device - use MPS for Apple Silicon, CUDA for NVIDIA, otherwise CPU
            if torch.backends.mps.is_available():
                self.device = 'mps'
                print(f"[ToxicityDetector] Using Apple Silicon GPU (MPS)")
            elif torch.cuda.is_available():
                self.device = 'cuda'
                print(f"[ToxicityDetector] Using NVIDIA GPU (CUDA)")
            else:
                self.device = 'cpu'
                print(f"[ToxicityDetector] Using CPU")
            
            # Load model with explicit device
            # The model will be cached in ~/.cache/huggingface/ after first download
            self.model = Detoxify('original', device=self.device)
            
            ToxicityDetector._initialized = True
            print("[ToxicityDetector] Model loaded successfully!")

    def score(self, text: str) -> Tuple[Dict[str, float], str]:
        """
        Returns:
            - Dict[str, float]: score per toxicity category (0-1 range)
            - str: predicted label (category with highest score)
        """
        
        # Get predictions from detoxify
        results = self.model.predict(text)
        
        # Round scores to 4 decimal places
        scores = {
            label: round(float(results[label]), 4)
            for label in self.LABELS
        }
        
        # Get the label with highest score
        predicted_label = max(scores, key=scores.get)
        
        return scores, predicted_label


In [3]:
def categorize_toxicity(scores: dict) -> str:
    """
    Categorize toxicity scores into levels using multiple dimensions.
    
    Logic:
    - Super_Toxic: severe_toxicity >= 0.5 OR toxicity >= 0.85
    - Toxic: toxicity >= 0.6 OR threat >= 0.6 OR identity_attack >= 0.6
    - Rude: toxicity >= 0.4 OR insult >= 0.5 OR obscene >= 0.5
    - Neutral: toxicity >= 0.15
    - Friendly: toxicity < 0.15
    """
    # Extract individual scores
    toxicity = scores.get("toxicity", 0.0)
    severe_toxicity = scores.get("severe_toxicity", 0.0)
    threat = scores.get("threat", 0.0)
    insult = scores.get("insult", 0.0)
    obscene = scores.get("obscene", 0.0)
    identity_attack = scores.get("identity_attack", 0.0)
    
    # Super Toxic: severe toxicity detected OR very high overall toxicity
    if severe_toxicity >= 0.5 or toxicity >= 0.85:
        return "Super_Toxic"
    
    # Toxic: high toxicity or high threat/identity attack
    if toxicity >= 0.6 or threat >= 0.6 or identity_attack >= 0.6:
        return "Toxic"
    
    # Rude: moderate toxicity or elevated insult/obscene language
    if toxicity >= 0.4 or insult >= 0.5 or obscene >= 0.5:
        return "Rude"
    
    # Neutral: mild toxicity present
    if toxicity >= 0.15:
        return "Neutral"
    
    # Friendly: very low toxicity across all categories
    return "Friendly"


def analyze_toxicity(text: str):
    """Analyze toxicity level of text using Detoxify ToxicityDetector."""
    detector = ToxicityDetector()
    toxicity_scores, predicted_label = detector.score(text)
    
    # Get category based on all scores
    category = categorize_toxicity(toxicity_scores)

    return {
        "text": text[:100] + "..." if len(text) > 100 else text,
        "category": category,
        "overall_toxicity": toxicity_scores["toxicity"],
        "most_toxic_type": predicted_label,
        "all_scores": toxicity_scores
    }


In [4]:
# Test with different toxicity levels
examples = {
    "friendly": "This is such a wonderful and helpful article! Thank you for sharing.",
    "neutral": "The conference will be held on March 15, 2026 at 10:00 AM.",
    "rude": "That's a stupid idea and you should know better.",
    "toxic": "You're an idiot if you believe this garbage!",
    "super_toxic": "I hope something terrible happens to you, you worthless piece of trash!"
}

print("=== Toxicity Analysis Examples ===\n")
for level, text in examples.items():
    result = analyze_toxicity(text)
    print(f"{level.upper()}:")
    print(f"  Text: {text}")
    print(f"  Category: {result['category']}")
    print(f"  Overall Toxicity Score: {result['overall_toxicity']:.4f}")
    print(f"  Most Toxic Type: {result['most_toxic_type']}")
    print(f"  All Scores: {result['all_scores']}")
    print()


=== Toxicity Analysis Examples ===

FRIENDLY:
  Text: This is such a wonderful and helpful article! Thank you for sharing.
  Category: Friendly
  Overall Toxicity Score: 0.0006
  Most Toxic Type: toxicity
  All Scores: {'toxicity': 0.0006, 'severe_toxicity': 0.0001, 'obscene': 0.0002, 'threat': 0.0001, 'insult': 0.0002, 'identity_attack': 0.0001}

NEUTRAL:
  Text: The conference will be held on March 15, 2026 at 10:00 AM.
  Category: Friendly
  Overall Toxicity Score: 0.0006
  Most Toxic Type: toxicity
  All Scores: {'toxicity': 0.0006, 'severe_toxicity': 0.0001, 'obscene': 0.0002, 'threat': 0.0001, 'insult': 0.0002, 'identity_attack': 0.0001}

RUDE:
  Text: That's a stupid idea and you should know better.
  Category: Toxic
  Overall Toxicity Score: 0.8495
  Most Toxic Type: toxicity
  All Scores: {'toxicity': 0.8495, 'severe_toxicity': 0.0019, 'obscene': 0.1507, 'threat': 0.0013, 'insult': 0.1574, 'identity_attack': 0.0013}

TOXIC:
  Text: You're an idiot if you believe this garbage!


## Using the Toxicity Model from the Project

The actual implementation is in `models/toxicity/toxicity.py`. Let's import and use it:


In [5]:
import sys
sys.path.append('/Users/danielbirman/dev/capstone_factuality_factors')

from models.toxicity.toxicity import Toxicity

# Initialize the toxicity model
toxicity_model = Toxicity()

# Test text
test_text = "You're an idiot if you believe this garbage!"

# Get probability scores
scores = toxicity_model.probability(test_text)
print(f"Toxicity scores for: '{test_text}'\n")
print(scores)
print()

# Get category
category = toxicity_model.categorize(test_text)
print(f"Category: {category}")


Toxicity scores for: 'You're an idiot if you believe this garbage!'

{'toxicity': 0.9897, 'severe_toxicity': 0.0388, 'obscene': 0.7554, 'threat': 0.0012, 'insult': 0.9481, 'identity_attack': 0.0131}

Category: Super_Toxic


## Detoxify Model Options

Detoxify provides three pre-trained models:

1. **'original'** - Trained on Wikipedia comments (default in our implementation)
2. **'unbiased'** - Trained on additional data to reduce bias
3. **'multilingual'** - Supports multiple languages

You can change the model in `toxicity_model.py` by modifying the `Detoxify('original')` line.


In [6]:
# Compare different Detoxify models (optional - only if you want to test alternatives)
test_text = "This is a test of toxicity detection."

models = ['original', 'unbiased', 'multilingual']

for model_name in models:
    print(f"\n=== Using '{model_name}' model ===")
    model = Detoxify(model_name)
    results = model.predict(test_text)
    
    # Round scores for consistency
    scores = {k: round(float(v), 4) for k, v in results.items()}
    
    print(f"Text: {test_text}")
    print(f"Toxicity: {results['toxicity']:.4f}")
    print(f"Category: {categorize_toxicity(scores)}")



=== Using 'original' model ===
Text: This is a test of toxicity detection.
Toxicity: 0.0007
Category: Friendly

=== Using 'unbiased' model ===
Text: This is a test of toxicity detection.
Toxicity: 0.0004
Category: Friendly

=== Using 'multilingual' model ===
Text: This is a test of toxicity detection.
Toxicity: 0.0005
Category: Friendly


## Comprehensive Test with All Toxicity Categories

Test various types of toxic content to see how each category is detected:


In [7]:
detector = ToxicityDetector()

comprehensive_examples = [
    ("Friendly", "This is a wonderful article with great insights!"),
    ("Neutral", "The meeting is scheduled for 3 PM tomorrow."),
    ("Obscene", "This is some really f***ed up sh*t."),
    ("Threat", "I know where you live and I'm coming for you."),
    ("Insult", "You're such a pathetic loser and everyone knows it."),
    ("Identity Attack", "All people from that group are criminals and terrorists."),
]

print("=== Comprehensive Toxicity Analysis ===\n")
for label, text in comprehensive_examples:
    scores, predicted = detector.score(text)
    category = categorize_toxicity(scores)
    
    print(f"Expected Type: {label}")
    print(f"  Text: {text}")
    print(f"  Categorized as: {category}")
    print(f"  Overall Toxicity: {scores['toxicity']:.4f}")
    print(f"  Highest Score Type: {predicted} ({scores[predicted]:.4f})")
    
    # Show top 3 scores
    top_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:3]
    print(f"  Top 3 Scores: {', '.join([f'{k}={v:.4f}' for k, v in top_scores])}")
    print()


=== Comprehensive Toxicity Analysis ===

Expected Type: Friendly
  Text: This is a wonderful article with great insights!
  Categorized as: Friendly
  Overall Toxicity: 0.0006
  Highest Score Type: toxicity (0.0006)
  Top 3 Scores: toxicity=0.0006, obscene=0.0002, insult=0.0002

Expected Type: Neutral
  Text: The meeting is scheduled for 3 PM tomorrow.
  Categorized as: Friendly
  Overall Toxicity: 0.0006
  Highest Score Type: toxicity (0.0006)
  Top 3 Scores: toxicity=0.0006, obscene=0.0002, insult=0.0002

Expected Type: Obscene
  Text: This is some really f***ed up sh*t.
  Categorized as: Super_Toxic
  Overall Toxicity: 0.9914
  Highest Score Type: toxicity (0.9914)
  Top 3 Scores: toxicity=0.9914, obscene=0.9700, insult=0.2769

Expected Type: Threat
  Text: I know where you live and I'm coming for you.
  Categorized as: Rude
  Overall Toxicity: 0.4734
  Highest Score Type: toxicity (0.4734)
  Top 3 Scores: toxicity=0.4734, threat=0.3639, identity_attack=0.0118

Expected Type: Insult

## Understanding the Multi-Dimensional Categorization

The new categorization logic considers all toxicity dimensions:

- **Super_Toxic**: Triggered by `severe_toxicity >= 0.5` OR `toxicity >= 0.85`
- **Toxic**: Triggered by `toxicity >= 0.6` OR `threat >= 0.6` OR `identity_attack >= 0.6`
- **Rude**: Triggered by `toxicity >= 0.4` OR `insult >= 0.5` OR `obscene >= 0.5`
- **Neutral**: Triggered by `toxicity >= 0.15`
- **Friendly**: Everything else (toxicity < 0.15)

This means text can be categorized as "Rude" even if overall toxicity is moderate, as long as insults or obscene language are present.


In [8]:
# Demonstrate how different dimensions affect categorization
detector = ToxicityDetector()

dimension_examples = [
    "This is absolutely horrible and you're a complete moron!",  # High insult -> Rude/Toxic
    "I'm going to find you and hurt you.",  # High threat -> Toxic
    "Those people from that country are all criminals.",  # High identity_attack -> Toxic
    "What a bunch of bullsh*t!",  # High obscene -> Rude
]

print("=== Multi-Dimensional Categorization Examples ===\n")
for text in dimension_examples:
    scores, _ = detector.score(text)
    category = categorize_toxicity(scores)
    
    print(f"Text: {text}")
    print(f"Category: {category}")
    print(f"Scores breakdown:")
    for key, value in sorted(scores.items(), key=lambda x: x[1], reverse=True):
        print(f"  - {key}: {value:.4f}")
    print()


=== Multi-Dimensional Categorization Examples ===

Text: This is absolutely horrible and you're a complete moron!
Category: Super_Toxic
Scores breakdown:
  - toxicity: 0.9807
  - insult: 0.9273
  - obscene: 0.6892
  - severe_toxicity: 0.0260
  - identity_attack: 0.0104
  - threat: 0.0011

Text: I'm going to find you and hurt you.
Category: Toxic
Scores breakdown:
  - toxicity: 0.7994
  - threat: 0.7431
  - severe_toxicity: 0.0506
  - insult: 0.0488
  - obscene: 0.0354
  - identity_attack: 0.0221

Text: Those people from that country are all criminals.
Category: Friendly
Scores breakdown:
  - toxicity: 0.0577
  - insult: 0.0023
  - identity_attack: 0.0023
  - obscene: 0.0007
  - threat: 0.0007
  - severe_toxicity: 0.0002

Text: What a bunch of bullsh*t!
Category: Super_Toxic
Scores breakdown:
  - toxicity: 0.9533
  - obscene: 0.8066
  - insult: 0.7288
  - severe_toxicity: 0.0268
  - identity_attack: 0.0197
  - threat: 0.0012

